# l03 · 1일차 3교시 · 위키백과에 같은 4단계
**목표**: 서버만 바꿔서 l02의 4단계를 그대로 다시 씁니다. 여기에 두 가지 안전장치를 더합니다.

| 새로 더하는 것 | 왜 필요한가 |
|---|---|
| **User-Agent** 이름표 | 누가 요청하는지 서버에 알림. 위키백과는 이름표 없는 요청을 거부할 수 있음 |
| **try·except** 오류 처리 | 요청이 실패해도 프로그램이 멈추지 않게 |
| **get_summary 함수** | 4단계를 묶어 제목만 바꿔 부르기 |

> **노트북 사용법** · 셀을 클릭하고 **Shift+Enter**를 누르면 실행되고 다음 셀로 넘어갑니다.
> 셀 왼쪽의 `[1] [2]…`는 **실행한 순서**입니다. 위에서부터 차례로 실행하세요. 중간 셀을 건너뛰면 `NameError`가 납니다.
> 처음부터 깨끗하게 다시 하려면 위쪽 메뉴의 **Restart → Run All**을 누릅니다.

## 1. 위키백과 요약 API
날씨는 `?` 뒤 **파라미터**로 요청했지만, 위키백과는 **주소 끝에 문서 제목**을 붙여 요청합니다.

`https://ko.wikipedia.org/api/rest_v1/page/summary/` + `파이썬`

응답에서 쓸 키는 세 가지입니다.

| 키 | 뜻 |
|---|---|
| `title` | 문서 제목 |
| `description` | 한 줄 설명 (없는 문서도 있음) |
| `extract` | 문서 첫머리 요약문 |

In [1]:
import requests

base = "https://ko.wikipedia.org/api/rest_v1/page/summary/"
url = base + "파이썬"                                          # ① 주소 만들기: 끝에 제목을 붙임
headers = {"User-Agent": "APIClass/1.0 (me@example.com)"}      # 내 이름표 (아래에서 설명)

res = requests.get(url, headers=headers)                       # ② 보내기
print(res.status_code)                                         # ③ 확인하기

data = res.json()                                              # ④ 꺼내기
print(data["title"])
print(data.get("description", ""))   # .get(키, 기본값): 키가 없어도 오류 대신 기본값을 돌려줌
print(data["extract"])

200
파이썬
범용 고급 프로그래밍 언어
파이썬(영어: Python)은 1991년 네덜란드계 소프트웨어 엔지니어인 귀도 반 로섬이 발표한 고급 프로그래밍 언어로, '인터프리터를 사용하는 객체지향 언어'이자 플랫폼에 독립적인, 동적 타이핑(dynamically typed) 대화형 언어다. 파이썬이라는 이름은 귀도가 좋아하는 코미디인〈Monty Python's Flying Circus〉에서 따온 것이다. 이름에서 고대신화에 나오는 커다란 뱀을 연상하겠지만 이와는 무관하다. 다만 로고에는 뱀 두마리가 형상화되어 있다. 간결하고 읽기 쉬운 문법이 특징인 프로그래밍 언어로 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 활용된다.


> **`data["키"]`와 `data.get("키", 기본값)`의 차이**
> `data["description"]`은 키가 없으면 `KeyError`로 멈춥니다. `data.get("description", "")`은 키가 없으면 `""`를 돌려주고 계속 진행합니다. 있을 수도 없을 수도 있는 값에는 `.get()`을 씁니다.

## 2. 헤더와 User-Agent · 요청에 붙이는 내 이름표
식당 주문서 맨 위에 손님 이름과 연락처를 적어 두면, 주방은 누가 주문했는지 알고 문제가 생기면 연락할 수 있습니다.
**User-Agent**는 요청 헤더에 담아 보내는 "요청한 프로그램의 이름"입니다.

- 모양: `"프로그램이름/버전 (연락처)"`
- `params`처럼 **딕셔너리**로 만들어 `headers=`에 넘깁니다.

In [ ]:
# 이름표를 정하지 않으면 자동으로 붙는 값
print(requests.utils.default_user_agent())

# 내 이름표를 붙여 요청하고, 실제로 붙었는지 확인
headers = {"User-Agent": "APIClass/1.0 (me@example.com)"}
res = requests.get(url, headers=headers)
print(res.request.headers["User-Agent"])   # res.request = 내가 보낸 요청

> **자주 하는 실수** · `"UserAgent"`처럼 하이픈을 빼면 전혀 다른 헤더가 됩니다. 반드시 `"User-Agent"`.

## 3. 요청 오류 처리 · try·except
없는 문서를 요청하면 서버가 **404**를 보냅니다. 그런데 requests는 404를 받아도 **멈추지 않습니다**.
그대로 `res.json()["extract"]`를 꺼내면 `KeyError`로 프로그램이 멈춥니다.

그래서 세 가지를 함께 씁니다.

| 코드 | 하는 일 |
|---|---|
| `timeout=5` | 5초 안에 응답이 없으면 기다리지 않고 예외를 일으킴 |
| `res.raise_for_status()` | **③ 확인하기**: 4xx·5xx이면 예외를 일으킴 |
| `except requests.exceptions.RequestException:` | 요청 관련 실패를 한 곳에서 모두 받음 |

어떤 실패든 except 한 곳으로 모입니다.

| 상황 | 일어나는 일 | 결과 |
|---|---|---|
| 200 성공 | `raise_for_status()` 통과 | 요약문 |
| 404 없는 제목 | `raise_for_status()`가 예외 | "설명 없음" |
| 5초 넘게 응답 없음 | `timeout`이 예외 | "설명 없음" |
| 인터넷 끊김 | `requests.get()`이 예외 | "설명 없음" |

In [ ]:
url = base + "없는문서제목"
try:
    res = requests.get(url, headers=headers, timeout=5)
    print("상태 코드:", res.status_code)   # 404
    res.raise_for_status()                 # 404라서 여기서 예외 발생 → 바로 except로 점프
    summary = res.json()["extract"]        # 이 줄은 실행되지 않음
except requests.exceptions.RequestException:
    summary = "설명 없음"                   # 실패했을 때 쓸 값
print(summary)

## 4. get_summary 함수 · 4단계를 함수로 묶기
4단계를 함수 하나로 묶어 두면, 제목만 바꿔 `get_summary("뮤지컬")`처럼 한 줄로 부를 수 있습니다.
어떤 경우든 **문자열이 돌아오기 때문에** 반복문 안에서 써도 멈추지 않습니다.

| 줄 | 4단계 | 설명 |
|---|---|---|
| `url = BASE + title.replace(" ", "_")` | ① 주소 | 위키백과 주소는 띄어쓰기 대신 `_`를 씀 |
| `requests.get(url, headers=…, timeout=5)` | ② 보내기 | 5초까지만 기다림 |
| `res.raise_for_status()` | ③ 확인 | 4xx·5xx면 예외 → except |
| `return "설명 없음"` | | 실패하면 정해 둔 문자열 |
| `return res.json().get("extract", …)` | ④ 꺼내기 | 성공하면 요약문 |

변수 이름 `BASE`, `HEADERS`를 대문자로 쓴 것은 "바뀌지 않는 설정값"이라는 표시입니다 (파이썬 관례).

In [ ]:
import time
import requests

BASE = "https://ko.wikipedia.org/api/rest_v1/page/summary/"
HEADERS = {"User-Agent": "APIClass/1.0 (me@example.com)"}

def get_summary(title):
    url = BASE + title.replace(" ", "_")                       # ① 주소 만들기
    try:
        res = requests.get(url, headers=HEADERS, timeout=5)   # ② 보내기
        res.raise_for_status()                                # ③ 확인하기
    except requests.exceptions.RequestException:
        return "설명 없음"                                     # 실패하면 여기서 함수 끝
    return res.json().get("extract", "설명 없음")               # ④ 꺼내기

for word in ["파이썬", "없는용어123"]:
    print(word, ":", get_summary(word))
    time.sleep(0.5)   # 요청 사이에 0.5초 쉬어 서버 부담을 줄임 (예의)

## 5. [직접 해 보기] 내가 고른 단어 세 개 · 5분
리스트의 단어만 바꿔 보세요. 위키백과 문서 제목과 같은 단어면 요약문이, 없는 단어면 "설명 없음"이 나오고 반복은 계속됩니다.
영화·공연과 관련된 단어를 고르면 실습 과제의 예행연습이 됩니다.

In [ ]:
words = ["뮤지컬", "판소리", "없는용어123"]   # ← 원하는 단어로 바꿔 보세요
for word in words:
    print(word, ":", get_summary(word))
    time.sleep(0.5)

## 6. API 문서 읽기 · 메뉴판에서 찾을 네 가지
새 API를 만나면 문서에서 이 네 가지를 찾습니다. 찾은 다음에는 오늘과 같은 4단계로 코드를 씁니다.

| 찾을 것 | 오늘 (날씨, 위키백과) | 2일차 (KOBIS 영화, KOPIS 공연) |
|---|---|---|
| ① 주소 | `…/v1/forecast`, `…/summary/{제목}` | 박스오피스, 공연 목록 기능의 주소 |
| ② 파라미터 | 위도·경도, 경로 끝의 제목 | 키를 넣는 파라미터 이름, 날짜 형식 |
| ③ 응답 예시 | `current`, `title`, `extract` | 영화 제목, 공연 이름이 담긴 항목 이름 |
| ④ 한도 | 키 없음, 이용 규칙 | 키 필요, 하루 호출 한도 |

## 3교시 확인 퀴즈
**Q1.** 무엇이 출력될까요?
```python
t = "클래식 음악"
print(t.replace(" ", "_"))
```

**Q2.** 상태 코드가 404일 때 출력은? (res는 없는 제목으로 요청한 응답)
```python
try:
    print("A")
    res.raise_for_status()
    print("B")
except requests.exceptions.RequestException:
    print("C")
print("D")
```

**Q3.** 바르게 쓴 헤더는? ① `{"UserAgent": "App/1.0"}` ② `{"User-Agent": "App/1.0"}` ③ `{User-Agent: "App/1.0"}`

<details><summary>정답 보기</summary>

Q1 `클래식_음악` · Q2 `A`, `C`, `D` (B는 건너뜀) · Q3 ②
</details>

➡️ 다음 파일: **l04_실습_pandas정리.ipynb**